# Encounter class from cost and duration

Trains on whatever layer the pipeline view mounts — **gold**, a source's
**silver**, or its **bronze** — and predicts `encounter_class` (ambulatory,
emergency, inpatient, wellness…) from what an encounter cost and how long it
lasted.

Made for the hospital source's `encounters` extraction. With another table,
change `TARGET_COLUMN` in the data cell; the notebook stops with the list of
columns it found if the target is not there, rather than guessing one.

Training the same notebook on bronze and on silver is the point of allowing
bronze: the gap between the two accuracies is what the cleaning standard is
worth to this model.

In [ ]:
# Papermill injected parameters (do not edit this cell manually)
# These are overwritten at runtime by the pipeline.

DATASET_PATH = ""           # The layer chosen in the pipeline view, as one Parquet file
MODEL_OUTPUT_PATH = ""      # Where the trained model must be saved
PIPELINE_ID = ""            # UUID of the current pipeline run
MLFLOW_TRACKING_URI = ""    # MLflow tracking server URL
TRAINING_FEATURES_PATH = "" # Where to write the frame the model was fitted on

In [ ]:
import os

MODEL_NAME = "encounter-class-classifier"
VERSION = "1"

MODEL_OUTPUT_PATH = MODEL_OUTPUT_PATH or "./model.joblib"
PIPELINE_ID = PIPELINE_ID or "local-dev"
MLFLOW_TRACKING_URI = MLFLOW_TRACKING_URI or os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")

print(f"Model: {MODEL_NAME} v{VERSION}")
print(f"Dataset: {DATASET_PATH or '(none)'}")

In [ ]:
from pathlib import Path

import pandas as pd

TARGET_COLUMN = "encounter_class"

if not DATASET_PATH or not Path(DATASET_PATH).exists():
    raise ValueError(
        "DATASET_PATH is empty. Drag gold or a source's silver onto Input in "
        "the pipeline view, then run again."
    )

# gold-v0003.parquet, silver-source-2.parquet, bronze-source-2.parquet: the
# file name says which layer this run was given.
LAYER = Path(DATASET_PATH).name.split("-")[0]
df = pd.read_parquet(DATASET_PATH)

print(f"Layer: {LAYER} | rows: {len(df):,} | columns: {df.shape[1]}")
print(df.dtypes.to_string())

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"No {TARGET_COLUMN!r} column in this dataset. Columns: {list(df.columns)}. "
        "Set TARGET_COLUMN in the data cell to the label you want to predict."
    )

data = df[df[TARGET_COLUMN].notna()].copy()
y = data[TARGET_COLUMN].astype(str).str.strip().str.lower()

# How long the encounter lasted. Silver has these as timestamps already;
# bronze has them as text, so both go through to_datetime.
features = pd.DataFrame(index=data.index)
if {"started_at", "stopped_at"} <= set(data.columns):
    started = pd.to_datetime(data["started_at"], errors="coerce", utc=True)
    stopped = pd.to_datetime(data["stopped_at"], errors="coerce", utc=True)
    features["duration_minutes"] = (stopped - started).dt.total_seconds() / 60

# The money columns. In bronze they are text, so a number that will not parse
# becomes NaN — which is exactly the cost of not cleaning, made measurable.
for column in ("base_cost", "total_claim_cost", "payer_coverage"):
    if column in data.columns:
        features[column] = pd.to_numeric(data[column], errors="coerce")

# Left out on purpose: ids say nothing, and encounter_code / encounter_text
# name the encounter type outright — training on them would be reading the
# answer, not predicting it.
if features.empty:
    raise ValueError(f"None of the expected feature columns are here: {list(df.columns)}")

features = features.replace([np.inf, -np.inf], np.nan)
MISSING_RATE = float(features.isna().mean().mean())
X = features.fillna(features.median(numeric_only=True)).fillna(0)
FEATURE_NAMES = list(X.columns)

counts = y.value_counts()
stratify = y if counts.min() >= 2 else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=stratify
)

print(f"Features: {FEATURE_NAMES}")
print(f"Missing values filled: {MISSING_RATE:.1%}")
print(f"Classes:\n{counts.to_string()}")
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight="balanced",  # emergency and inpatient are rare next to ambulatory
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

importance = pd.Series(model.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=False)
print("Feature importance:")
print(importance.round(3).to_string())

In [ ]:
import mlflow
from sklearn.metrics import accuracy_score, classification_report, f1_score

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")

print(classification_report(y_test, y_pred, zero_division=0))
print(f"Accuracy: {accuracy:.4f} | F1 (weighted): {f1:.4f} | layer: {LAYER}")

# Log into the run the pipeline opened (it injects MLFLOW_RUN_ID), or into a
# new one when the notebook runs by hand.
RUN_ID = globals().get("MLFLOW_RUN_ID") or ""
started_here = mlflow.active_run() is None
if started_here:
    if RUN_ID:
        mlflow.start_run(run_id=RUN_ID)
    else:
        mlflow.start_run(run_name=f"{MODEL_NAME}-{PIPELINE_ID}")

mlflow.log_param("layer", LAYER)
mlflow.log_param("dataset_file", Path(DATASET_PATH).name)
mlflow.log_param("target_column", TARGET_COLUMN)
mlflow.log_param("features", ",".join(FEATURE_NAMES))
mlflow.log_metric("accuracy", accuracy)
mlflow.log_metric("f1_weighted", f1)
mlflow.log_metric("missing_rate", MISSING_RATE)
mlflow.log_metric("n_samples", float(len(X)))

if started_here:
    mlflow.end_run()

In [ ]:
import joblib

joblib.dump(model, MODEL_OUTPUT_PATH)
print(f"Model saved to {MODEL_OUTPUT_PATH}")

# The exact frame passed to fit(), same columns in the same order. The
# platform reads it to tell the model's test page which inputs to send, in
# what range, and what the model answers across them.
if TRAINING_FEATURES_PATH:
    X_train.to_parquet(TRAINING_FEATURES_PATH)
    print(f"Training features ({X_train.shape[0]:,} rows) written for the model card.")